In [1]:
import json
from pathlib import Path
from typing import Optional, Tuple
import pycountry

In [2]:
CODESET_CACHE: dict = {}
VALID_TERM_UNITS = {"DAYS", "WEEK", "MNTH", "YEAR"}
VALID_DEBT_SENIORITY = {"SNDB", "SBOD", "JUND", "OTHR"}
VALID_DELIVERY_TYPE = {"CASH", "PHYS", "OTHR"}

CODESET_BASE = Path(r"C:/Users/30222/Desktop/ANNA-DSB/product_definitions/PROD/OTC-Products/codesets")
UPI_BASE     = Path(r"C:/Users/30222/Desktop/ANNA-DSB/product_definitions/PROD/OTC-Products/UPI")

In [3]:
def load_codeset(codeset_name: str) -> list:
    if codeset_name in CODESET_CACHE:
        return CODESET_CACHE[codeset_name]

    codeset_path = CODESET_BASE / codeset_name
    if codeset_path.exists():
        with open(codeset_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            codes = data
        elif isinstance(data, dict):
            codes = (data.get("codes") or data.get("values")
                     or list(data.keys()))
        else:
            codes = []
        CODESET_CACHE[codeset_name] = codes
    else:
        CODESET_CACHE[codeset_name] = []

    return CODESET_CACHE[codeset_name]

class UpiLookupResult:
    def __init__(
        self,
        trade_id: str,
        status: str,
        matched_template: Optional[str] = None,
        upi_code: Optional[str] = None,
        classification_note: Optional[str] = None,
        validation_errors: list = None,
        warnings: list = None,
    ):
        self.trade_id           = trade_id
        self.status             = status
        self.matched_template   = matched_template
        self.upi_code           = upi_code
        self.classification_note = classification_note
        self.validation_errors  = validation_errors or []
        self.warnings           = warnings or []

    def to_dict(self) -> dict:
        return {
            "trade_id":           self.trade_id,
            "status":             self.status,
            "matched_template":   self.matched_template,
            "upi_code":           self.upi_code,
            "classification_note": self.classification_note,
            "validation_errors":  self.validation_errors,
            "warnings":           self.warnings,
        }

In [4]:
def find_product_template(
    asset_class: str, instrument_type: str, use_case: str
) -> Optional[dict]:
    if not asset_class:
        return None

    asset_dir = UPI_BASE / asset_class
    if not asset_dir.exists() or not asset_dir.is_dir():
        return None

    instr_lower    = instrument_type.lower() if instrument_type else ""
    usecase_lower  = use_case.lower()        if use_case        else ""

    for file_path in asset_dir.glob("*.json"):
        stem_lower = file_path.stem.lower()
        if instr_lower and usecase_lower:
            if instr_lower in stem_lower and usecase_lower in stem_lower:
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        template = json.load(f)
                    template["_file_stem"] = file_path.stem
                    return template
                except Exception:
                    continue

    return None

In [5]:
def validate_attributes(trade: dict, template: dict) -> Tuple[list, list]:
    errors:   list = []
    warnings: list = []

    for ccy_field in ("notional_currency", "notional_currency_leg1",
                      "notional_currency_leg2", "settlement_currency"):
        ccy = trade.get(ccy_field)
        if ccy:
            is_valid = (pycountry.currencies.get(alpha_3=ccy) is not None
                        or ccy == "XAU")
            if not is_valid:
                errors.append(f"Invalid currency code in '{ccy_field}': {ccy}")

    for rate_field in ("reference_rate", "reference_rate_leg1",
                       "reference_rate_leg2"):
        ref_rate = trade.get(rate_field)
        if ref_rate:
            if "LIBOR" in ref_rate.upper():
                warnings.append(
                    f"Deprecated rate code in '{rate_field}': {ref_rate}. "
                    "LIBOR ceased June 2023; legacy trades remain valid under prior reporting rules."
                )
            else:
                valid_rates = load_codeset("FpmlRatesReferenceRate.json")
                if valid_rates and ref_rate not in valid_rates:
                    errors.append(
                        f"Invalid reference rate code in '{rate_field}': {ref_rate}"
                    )

    for unit_field in ("reference_rate_term_unit",
                       "reference_rate_term_leg1_unit",
                       "reference_rate_term_leg2_unit"):
        term_unit = trade.get(unit_field)
        if term_unit and term_unit not in VALID_TERM_UNITS:
            errors.append(
                f"Invalid term unit in '{unit_field}': {term_unit}. "
                f"Expected one of {sorted(VALID_TERM_UNITS)}."
            )

    for val_field in ("reference_rate_term_value",
                      "reference_rate_term_leg1_value",
                      "reference_rate_term_leg2_value"):
        term_val = trade.get(val_field)
        if term_val is not None:
            try:
                if float(term_val) <= 0:
                    errors.append(
                        f"Term value out of range in '{val_field}': {term_val} (must be > 0)."
                    )
            except (TypeError, ValueError):
                errors.append(
                    f"Non-numeric term value in '{val_field}': {term_val}."
                )

    debt_seniority = trade.get("debt_seniority")
    if debt_seniority and debt_seniority not in VALID_DEBT_SENIORITY:
        errors.append(
            f"Invalid debt seniority: {debt_seniority}. "
            f"Expected one of {sorted(VALID_DEBT_SENIORITY)}."
        )

    delivery_type = trade.get("delivery_type")
    if delivery_type and delivery_type not in VALID_DELIVERY_TYPE:
        errors.append(
            f"Invalid delivery type: {delivery_type}. "
            f"Expected one of {sorted(VALID_DELIVERY_TYPE)}."
        )

    return errors, warnings


In [6]:
def lookup_upi(parsed_trade: dict) -> dict:
    trade_id          = parsed_trade.get("trade_id")
    classification_flag = parsed_trade.get("classification_flag", "")
    asset_class       = parsed_trade.get("asset_class", "")
    instrument_type   = parsed_trade.get("instrument_type", "")
    use_case          = parsed_trade.get("use_case", "")

    normalized = parsed_trade.get("normalized_trade", {}) or {}

    if classification_flag == "NOVEL_INSTRUMENT_NO_TAXONOMY":
        note = (
            f"Instrument '{instrument_type}' (use_case: {use_case}) under "
            f"asset class '{asset_class}' has no product definition in the "
            "ANNA-DSB UPI library. Binary event / prediction contracts are "
            "currently outside the OTC derivatives taxonomy in most "
            "jurisdictions (EMIR, CFTC Part 43/45). "
            "Refer to Module 4 for detailed regulatory classification analysis."
        )
        return UpiLookupResult(
            trade_id=trade_id,
            status="NO_PRODUCT_DEFINITION",
            classification_note=note,
        ).to_dict()

    template = find_product_template(asset_class, instrument_type, use_case)
    if not template:
        return UpiLookupResult(
            trade_id=trade_id,
            status="NOT_FOUND",
            classification_note=(
                f"No ANNA-DSB template found for "
                f"{asset_class}/{instrument_type}/{use_case}."
            ),
        ).to_dict()

    matched_template_name = template.get("_file_stem")

    errors, warnings = validate_attributes(normalized, template)

    if errors:
        return UpiLookupResult(
            trade_id=trade_id,
            status="INVALID_ATTRIBUTES",
            matched_template=matched_template_name,
            validation_errors=errors,
            warnings=warnings,
        ).to_dict()

    upi_code = (template.get("upi_code")
                or template.get("UPI")
                or template.get("upi")
                or None)

    return UpiLookupResult(
        trade_id=trade_id,
        status="FOUND",
        matched_template=matched_template_name,
        upi_code=upi_code,
        warnings=warnings,
    ).to_dict()

In [7]:
def process_trades(input_file: str, output_file: str) -> None:
    with open(input_file, "r", encoding="utf-8") as f:
        trades = json.load(f)

    results = [lookup_upi(trade) for trade in trades]

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"[Done] Processed {len(results)} trades -> {output_file}")

In [8]:
if __name__ == "__main__":
    process_trades(
        "c:/Users/30222/Desktop/group/outputs/module1_parser_output.json",
        "M2_output.json",
    )

[Done] Processed 33 trades -> M2_output.json
